In [ ]:
import numpy as np
import cobra
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import random
from node2vec import Node2Vec
from sklearn.cluster import KMeans
import networkx as nx
from scipy import sparse
import sys
sys.path.append('../../')
from helper_function import *
plt.rcParams['font.family'] = 'Arial'
cobra.Configuration().solver = 'glpk'

In [ ]:
model = cobra.io.load_matlab_model('../../universal_model.mat')
with open('../../FBAsoluition/M9_sufficient_423_1_proteome_weakbg_unimodel.pkl', 'rb') as pickle_file:
    data2 = pickle.load(pickle_file)
    all_total_nutrient_count2 = data2['all_total_nutrient_count']
    all_total_intake2 = data2['all_total_intake']
    all_growth_rate2 = data2['all_growth_rate']
    all_random_medium2 = data2['all_random_medium']
    all_met_flux2 = data2['all_met_flux']
    del data2

Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-29


In [3]:
S_df = cobra.util.create_stoichiometric_matrix(model, array_type='DataFrame')
reaction_list = list(S_df.columns)
metabolite_list = list(S_df.index)

In [ ]:
union_M = sparse.load_npz(
    "../../Coarse_graining_result/union_M.npz"
)
union_M = union_M.toarray()

In [ ]:
G_mfg = nx.DiGraph()
rxn_nodes = [f"{name}_fwd" for name in reaction_list] + [f"{name}_bwd" for name in reaction_list]

for i in range(len(rxn_nodes)):
    for j in range(len(rxn_nodes)):
        if union_M[i, j] > 1e-2:  # threshold to show
            G_mfg.add_edge(rxn_nodes[i], rxn_nodes[j], weight=round(union_M[i, j], 2))


In [ ]:
random.seed(42)
np.random.seed(42)

node2vec = Node2Vec(
    G_mfg,
    dimensions=64,
    walk_length=30,
    num_walks=200,
    workers=4,
    seed=42,
    p=1,
    q=1
)

model2 = node2vec.fit(window=10, min_count=1, batch_words=4, seed=42) 
nodes = list(G_mfg.nodes())

node_vectors = np.array([model2.wv[str(n)] for n in nodes])


/home/complicatedz/miniconda3/envs/autopacman/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating walks (CPU: 4): 100%|██████████| 50/50 [00:04<00:00, 10.56it/s]


In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=20, random_state=44)
labels = kmeans.fit_predict(node_vectors)

clustered_nodes = {i: [] for i in range(20)}
nodes = list(G_mfg.nodes())
for node, label in zip(nodes, labels):
    clustered_nodes[label].append(node)

/home/complicatedz/miniconda3/envs/autopacman/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
